# 01 — Dataset & Preprocessing (Sentinel-1 SAR → LULC labels)

Builds the training/test dataset directly on the server from Google Earth Engine:
Sentinel-1 SAR imagery (inputs) + ESA WorldCover (labels), for 6 training and 2
held-out test regions across all 12 months of 2021.

**7 classes:** Water, Trees, Grassland, Cropland, Built-up, Bare/Sparse, Wetland.

**Preprocessing per monthly composite**
1. Sentinel-1 GRD, IW mode, descending orbit, VV+VH (GRD is already calibrated to σ⁰ in dB and terrain-geocoded by ESA).
2. Monthly **median** composite (reduces speckle + fills revisit gaps).
3. **Speckle filtering**: 1.5-pixel circular focal median.
4. **dB normalization**: VV clipped to [-25, 0] dB → [0,1]; VH clipped to [-30, -5] dB → [0,1].
5. Labels: ESA WorldCover 2021 v200, remapped from 11 classes to our 7.
6. **Patchify** into 256×256 tiles; drop patches with >10% nodata or any NaN label.
7. Augmentation (flips + 90° rotations) is applied later, in training.

In [ ]:
import ee, geemap, numpy as np, rasterio
ee.Authenticate(auth_mode="notebook")   # one-time; paste the code
ee.Initialize(project="lulc-sentinel-513912")

YEAR = 2021; MONTHS = list(range(1,13)); SCALE = 10; PATCH = 256
CLASSES = ["Water","Trees","Grassland","Cropland","Built-up","Bare/Sparse","Wetland"]

REGIONS_TRAIN = {
    "po_valley":[9.00,45.15,9.25,45.35], "rhine_neckar":[8.40,49.40,8.65,49.58],
    "flevoland":[5.35,52.45,5.65,52.65], "andalusia":[-6.05,37.30,-5.80,37.50],
    "danube":[18.90,47.30,19.15,47.50],  "saxony":[12.20,51.25,12.50,51.45]}
REGIONS_TEST = {"beauce":[1.70,48.00,1.95,48.20], "catalonia":[1.90,41.45,2.20,41.65]}

In [ ]:
def s1_month(aoi, year, month):
    start = ee.Date.fromYMD(year, month, 1); end = start.advance(1, "month")
    col = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi).filterDate(start, end)
           .filter(ee.Filter.eq("instrumentMode","IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VV"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VH"))
           .filter(ee.Filter.eq("orbitProperties_pass","DESCENDING")).select(["VV","VH"]))
    img = col.median().focalMedian(1.5, "circle", "pixels")              # speckle filter
    vv = img.select("VV").clamp(-25, 0).unitScale(-25, 0)                # dB -> [0,1]
    vh = img.select("VH").clamp(-30, -5).unitScale(-30, -5)
    return vv.addBands(vh).rename(["VV","VH"]).toFloat()

def worldcover_labels(aoi):
    wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map").clip(aoi)
    # WorldCover -> our 7 classes (Water=0 Trees=1 Grass=2 Crop=3 Built=4 Bare=5 Wetland=6)
    frm = [80,10,20,30,40,50,60,90,95,70,100]; to = [0,1,2,2,3,4,5,6,0,5,2]
    return wc.remap(frm, to).rename("lulc").toInt()

In [ ]:
def patchify(x, y, patch=PATCH):
    """x:(H,W,2) SAR, y:(H,W) labels -> lists of clean 256x256 patches."""
    X, Y = [], []
    H, W = y.shape
    for i in range(0, H - patch + 1, patch):
        for j in range(0, W - patch + 1, patch):
            xp = x[i:i+patch, j:j+patch]; yp = y[i:i+patch, j:j+patch]
            if np.isnan(xp).mean() < 0.1 and not np.isnan(yp).any():     # drop nodata/NaN
                X.append(np.nan_to_num(xp, nan=0.0).astype(np.float32))
                Y.append(np.clip(yp, 0, 6).astype(np.uint8))
    return X, Y

# For each region x month: download the SAR+label stack, read, patchify, accumulate.
# Full loop is in server_data_pull.py; it writes train.npz and test.npz.

## Executed result (DataLab server)

```
Saved train.npz (6216, 256, 256, 2) and test.npz (2208, 256, 256, 2)

CLASS DISTRIBUTION (train):
  Water          43,751,290  (12.6%)
  Trees          77,328,984  (22.3%)
  Grassland      45,531,206  (13.1%)
  Cropland      106,972,360  (30.9%)
  Built-up       70,058,910  (20.2%)
  Bare/Sparse     1,416,603  ( 0.4%)
  Wetland         1,232,871  ( 0.4%)
```

**Split:** 6 training regions (5,284 train / 932 val after an 85/15 split) and 2
**geographically held-out** test regions (2,208 patches). Bare/Sparse and Wetland are
severely under-represented (0.4% each) — addressed with inverse-frequency class
weighting in training, and revisited in the critical discussion.